# Lesson 03 — Graph Profiling

## Learning Goal

Learn to diagnose graph structure systematically before running algorithms.

By the end of this lesson, you will:
- Compute fundamental graph metrics (degree, density, components)
- Visualize degree distributions and component sizes
- Identify anomalies: obsolete documents, ownerless nodes, isolated clusters, duplicates
- Understand how profiling catches data quality issues

**Duration**: 90 minutes

**Why this matters**: Algorithm results are only as good as your data. Profiling before analysis saves time and prevents misleading conclusions.

**Estimated time per section**:
- Setup & data: 10 minutes
- Graph construction: 10 minutes
- Profiling metrics: 30 minutes
- Anomaly detection: 20 minutes
- Subgraph analysis: 10 minutes
- Exercises: 10 minutes

## Setup and Data Loading

Import libraries and load the document-policy governance dataset.

In [1]:
import sys
from pathlib import Path

# Add src directory to path to import config
project_root = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load config or construct path manually
try:
    from src.config import Config
    DATA_DIR = Config.PROJECT_ROOT / 'data' / 'seed' / 'document_policy'
except ImportError:
    # Fallback: construct path from notebook location
    notebook_dir = Path.cwd()
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir.parent.parent if (notebook_dir.parent.parent / 'data').exists() else notebook_dir
    DATA_DIR = project_root / 'data' / 'seed' / 'document_policy'

def display_csv_head(df, name, n=5):
    """Display DataFrame info and head."""
    print(f"\n{name}:")
    print(f"  Shape: {df.shape} (rows, columns)")
    print(f"  Columns: {list(df.columns)}")
    print(f"\n  First {n} rows:")
    print(df.head(n).to_string(index=False))

# Load datasets
topics_df = pd.read_csv(DATA_DIR / 'topics.csv')
owners_df = pd.read_csv(DATA_DIR / 'owners.csv')
teams_df = pd.read_csv(DATA_DIR / 'teams.csv')
documents_df = pd.read_csv(DATA_DIR / 'documents.csv')
references_df = pd.read_csv(DATA_DIR / 'references.csv')
usages_df = pd.read_csv(DATA_DIR / 'usages.csv')

print("Data loaded successfully!")
print(f"  Topics: {len(topics_df)} rows")
print(f"  Owners: {len(owners_df)} rows")
print(f"  Teams: {len(teams_df)} rows")
print(f"  Documents: {len(documents_df)} rows")
print(f"  References: {len(references_df)} rows (edges)")
print(f"  Usages: {len(usages_df)} rows (team-document relationships)")

Data loaded successfully!
  Topics: 15 rows
  Owners: 10 rows
  Teams: 8 rows
  Documents: 100 rows
  References: 290 rows (edges)
  Usages: 200 rows (team-document relationships)


## 1. Understanding the Dataset

We have a **document governance network** representing how documents are managed in an organization:

**Entities**:
- **Documents** (100): Policies, procedures, guidelines with status (active/deprecated/draft)
- **Topics** (15): Categories (Compliance, Operations, Security, Finance, HR)
- **Owners** (10): People responsible for maintaining documents
- **Teams** (8): Groups that use/depend on documents

**Relationships**:
- **REFERENCES**: Document → Document (citations, weighted by usage count)
- **USES**: Team → Document (consumption/dependency)
- **ABOUT**: Document → Topic (classification)
- **OWNED_BY**: Document → Owner (governance)

**Why this matters**: Document networks reveal governance debt, single points of failure, and data quality issues.

In [ ]:
# Examine each dataset
display_csv_head(topics_df, "Topics", n=5)
display_csv_head(owners_df, "Owners", n=5)
display_csv_head(teams_df, "Teams", n=5)

In [ ]:
display_csv_head(documents_df, "Documents (first 10)", n=10)
display_csv_head(references_df, "References (edges)", n=5)
display_csv_head(usages_df, "Team Usage (edges)", n=5)

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print(f"\nDocuments by status:")
print(documents_df['status'].value_counts())
print(f"\nDocuments by topic:")
print(documents_df['topic_id'].value_counts().head(10))
print(f"\nOwned vs ownerless documents:")
print(f"  Owned: {documents_df['owner_id'].notna().sum()}")
print(f"  Ownerless: {documents_df['owner_id'].isna().sum()}")

## 2. Building the Graph

Transform relational data into a network. We'll focus on the **document-to-document reference network** as the primary graph, then layer in team usage.

In [2]:
# Create graph with documents as nodes
G = nx.Graph()

# Step 1: Add document nodes with attributes
for idx, row in documents_df.iterrows():
    G.add_node(
        row['document_id'],
        node_type='document',
        name=row['name'],
        status=row['status'],
        owner_id=row['owner_id'],
        topic_id=row['topic_id'],
        created_date=row['created_date'],
        modified_date=row['modified_date']
    )

print(f"Step 1: Added {len(documents_df)} document nodes")
print(f"  Total nodes: {G.number_of_nodes()}")

# Step 2: Add document-reference edges (weighted by usage count)
for idx, row in references_df.iterrows():
    from_doc = row['from_doc_id']
    to_doc = row['to_doc_id']
    weight = row['weight']
    G.add_edge(from_doc, to_doc, edge_type='references', weight=weight)

print(f"\nStep 2: Added {len(references_df)} document-reference edges")
print(f"  Total edges: {G.number_of_edges()}")

# Step 3: Add team nodes and usage edges (optional - for later analysis)
for idx, row in teams_df.iterrows():
    G.add_node(row['team_id'], node_type='team', name=row['name'], size=row['size'])

for idx, row in usages_df.iterrows():
    team_id = row['team_id']
    doc_id = row['document_id']
    usage_count = row['usage_count']
    G.add_edge(team_id, doc_id, edge_type='uses', usage_count=usage_count)

print(f"\nStep 3: Added {len(teams_df)} team nodes and {len(usages_df)} usage edges")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()}")

print(f"\n" + "="*60)
print(f"GRAPH SUMMARY")
print(f"="*60)
print(f"Nodes: {G.number_of_nodes()} (100 documents + 8 teams)")
print(f"Edges: {G.number_of_edges()} ({len(references_df)} references + {len(usages_df)} usages)")
print(f"Density: {nx.density(G):.4f}")

Step 1: Added 100 document nodes
  Total nodes: 100

Step 2: Added 290 document-reference edges
  Total edges: 290

Step 3: Added 8 team nodes and 200 usage edges
  Total nodes: 108
  Total edges: 490

GRAPH SUMMARY
Nodes: 108 (100 documents + 8 teams)
Edges: 490 (290 references + 200 usages)
Density: 0.0848


## 3. Graph Profiling: Computing Key Metrics

Before running algorithms, inspect the graph structure systematically. These metrics reveal fundamental properties and potential data quality issues.

In [3]:
# Helper function: Comprehensive graph profile
def graph_profile(G, verbose=True):
    """Generate a comprehensive profile of graph structure."""
    profile = {
        'nodes': G.number_of_nodes(),
        'edges': G.number_of_edges(),
        'density': nx.density(G),
        'avg_degree': sum(dict(G.degree()).values()) / G.number_of_nodes(),
        'min_degree': min(dict(G.degree()).values()),
        'max_degree': max(dict(G.degree()).values()),
        'components': nx.number_connected_components(G),
        'isolates': len(list(nx.isolates(G))),
    }
    
    # Only compute clustering if graph is undirected and small enough
    if G.number_of_nodes() < 10000:
        profile['avg_clustering'] = nx.average_clustering(G)
    
    if verbose:
        print("Graph Profile:")
        for key, value in profile.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
    
    return profile

# Compute profile
profile = graph_profile(G)

# Interpretation
print("\nInterpretation:")
print(f"  - Low density ({profile['density']:.4f}): Graph is sparse (documents loosely connected)")
print(f"  - {profile['components']} component(s): Graph connectivity level")
print(f"  - {profile['isolates']} isolate(s): Documents with no connections")
print(f"  - Avg clustering ({profile['avg_clustering']:.4f}): Local clustering tendency")

Graph Profile:
  nodes: 108
  edges: 490
  density: 0.0848
  avg_degree: 9.0741
  min_degree: 1
  max_degree: 35
  components: 1
  isolates: 0
  avg_clustering: 0.1951

Interpretation:
  - Low density (0.0848): Graph is sparse (documents loosely connected)
  - 1 component(s): Graph connectivity level
  - 0 isolate(s): Documents with no connections
  - Avg clustering (0.1951): Local clustering tendency


In [5]:
# Degree statistics
degrees = dict(G.degree())
degree_values = list(degrees.values())

print("Degree Distribution Statistics:")
print(f"  Min degree: {min(degree_values)}")
print(f"  Max degree: {max(degree_values)}")
print(f"  Mean degree: {np.mean(degree_values):.2f}")
print(f"  Median degree: {np.median(degree_values):.2f}")
print(f"  Std dev: {np.std(degree_values):.2f}")
print(f"  25th percentile: {np.percentile(degree_values, 25):.2f}")
print(f"  75th percentile: {np.percentile(degree_values, 75):.2f}")
print(f"  95th percentile: {np.percentile(degree_values, 95):.2f}")

Degree Distribution Statistics:
  Min degree: 1
  Max degree: 35
  Mean degree: 9.07
  Median degree: 8.00
  Std dev: 5.94
  25th percentile: 5.00
  75th percentile: 11.00
  95th percentile: 21.60


In [ ]:
# Visualize degree distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(degree_values, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(degree_values), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(degree_values):.2f}')
axes[0].axvline(np.median(degree_values), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(degree_values):.2f}')
axes[0].set_xlabel('Degree', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Degree Distribution (All Nodes)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_degrees = sorted(degree_values)
cumulative = np.arange(1, len(sorted_degrees) + 1) / len(sorted_degrees)
axes[1].plot(sorted_degrees, cumulative, marker='o', linestyle='-', linewidth=2, markersize=4, color='steelblue')
axes[1].set_xlabel('Degree', fontsize=12)
axes[1].set_ylabel('Cumulative Probability', fontsize=12)
axes[1].set_title('Cumulative Degree Distribution', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Degree distribution shape: {'Right-skewed (power-law)' if np.mean(degree_values) > np.median(degree_values) else 'Symmetric/Left-skewed'}")

In [ ]:
# Identify high-degree nodes (hubs)
high_degree_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:15]

print("Top 15 High-Degree Nodes (Hubs):")
print(f"\n{'Node ID':<12} {'Degree':<8} {'Status':<12} {'Owner':<12} {'Name'}")
print("-" * 70)

for node_id, degree in high_degree_nodes:
    node_data = G.nodes[node_id]
    if node_data.get('node_type') == 'document':
        status = node_data.get('status', 'N/A')
        owner = node_data.get('owner_id', 'NULL')
        name = node_data.get('name', 'N/A')[:35]
        print(f"{node_id:<12} {degree:<8} {status:<12} {str(owner):<12} {name}")

In [ ]:
# Connected components analysis
components = list(nx.connected_components(G))
component_sizes = [len(c) for c in components]
component_sizes.sort(reverse=True)

print(f"Connected Components: {len(components)}")
print(f"\nComponent sizes (top 10):")
for i, size in enumerate(component_sizes[:10]):
    pct = 100 * size / G.number_of_nodes()
    print(f"  Component {i+1}: {size} nodes ({pct:.1f}%)")

# Visualize component sizes
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(component_sizes)), component_sizes, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Component Index', fontsize=12)
ax.set_ylabel('Component Size (nodes)', fontsize=12)
ax.set_title('Connected Component Sizes', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nLargest component: {component_sizes[0]} nodes ({100*component_sizes[0]/G.number_of_nodes():.1f}% of graph)")
print(f"Single-node components (isolates): {sum(1 for s in component_sizes if s == 1)}")

In [ ]:
# Find and analyze isolate nodes
isolates = list(nx.isolates(G))

print(f"Isolate Nodes (Degree = 0): {len(isolates)}")
if len(isolates) > 0:
    print(f"\n{'Node ID':<12} {'Status':<12} {'Owner':<12} {'Name'}")
    print("-" * 50)
    for node_id in isolates[:10]:  # Show first 10
        node_data = G.nodes[node_id]
        if node_data.get('node_type') == 'document':
            status = node_data.get('status', 'N/A')
            owner = node_data.get('owner_id', 'NULL')
            name = node_data.get('name', 'N/A')[:35]
            print(f"{node_id:<12} {status:<12} {str(owner):<12} {name}")
    
    if len(isolates) > 10:
        print(f"... and {len(isolates) - 10} more")

# Why are there isolates?
print(f"\nWhy isolates might exist:")
print(f"  - New documents not yet referenced")
print(f"  - Archived documents no longer in use")
print(f"  - Data quality issues (missing relationships)")

In [ ]:
# Clustering coefficient analysis
clustering_coeffs = nx.clustering(G)
clustering_values = list(clustering_coeffs.values())

print(f"Clustering Coefficient Statistics:")
print(f"  Average clustering: {np.mean(clustering_values):.4f}")
print(f"  Min clustering: {np.min(clustering_values):.4f}")
print(f"  Max clustering: {np.max(clustering_values):.4f}")
print(f"  Median clustering: {np.median(clustering_values):.4f}")

# Visualize clustering distribution
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(clustering_values, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(np.mean(clustering_values), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(clustering_values):.4f}')
ax.set_xlabel('Local Clustering Coefficient', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Clustering Coefficient Distribution', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  - High clustering: Tight communities (documents that reference each other)")
print(f"  - Low clustering: Sparse network, few triangles (references are more tree-like)")

## 4. Anomaly Detection via Profiling

Use metrics to discover data quality issues and governance problems. These anomalies would cause misleading algorithm results if not addressed first.

In [6]:
# Anomaly 1: Obsolete but heavily used documents (governance debt)
print("ANOMALY 1: Obsolete But Heavily Used Documents")
print("These documents are marked 'deprecated' but still referenced frequently.")
print("This indicates governance debt: teams haven't migrated to replacements.\n")

deprecated_docs = documents_df[documents_df['status'] == 'deprecated']['document_id'].values
deprecated_with_degree = [
    (doc_id, degrees.get(doc_id, 0))
    for doc_id in deprecated_docs
]
deprecated_with_degree.sort(key=lambda x: x[1], reverse=True)

median_degree = np.median(list(degrees.values()))
obsolete_but_used = [
    (doc_id, degree) for doc_id, degree in deprecated_with_degree
    if degree > median_degree
]

if len(obsolete_but_used) > 0:
    print(f"Found {len(obsolete_but_used)} obsolete but heavily used documents (degree > {median_degree:.0f}):")
    print(f"\n{'Document ID':<12} {'Degree':<8} {'Name'}")
    print("-" * 60)
    for doc_id, degree in obsolete_but_used:
        doc_data = documents_df[documents_df['document_id'] == doc_id].iloc[0]
        name = doc_data['name'][:45]
        print(f"{doc_id:<12} {degree:<8} {name}")
    print(f"\n⚠️ GOVERNANCE RISK: Migration effort needed")
else:
    print("No obsolete but heavily used documents found.")

ANOMALY 1: Obsolete But Heavily Used Documents
These documents are marked 'deprecated' but still referenced frequently.
This indicates governance debt: teams haven't migrated to replacements.

Found 4 obsolete but heavily used documents (degree > 8):

Document ID  Degree   Name
------------------------------------------------------------
DOC_0019     14       Table - True
DOC_0096     11       Traditional - Yes
DOC_0057     10       Election - Common
DOC_0044     9        Top - Early

⚠️ GOVERNANCE RISK: Migration effort needed


In [ ]:
# Anomaly 2: Ownerless critical documents (accountability risk)
print("\nANOMOLY 2: Ownerless Critical Documents")
print("These documents have high degree (many references) but no assigned owner.")
print("This indicates accountability risk: unclear who maintains these docs.\n")

ownerless_docs = documents_df[documents_df['owner_id'].isnull()]['document_id'].values
ownerless_with_degree = [
    (doc_id, degrees.get(doc_id, 0))
    for doc_id in ownerless_docs
]
ownerless_with_degree.sort(key=lambda x: x[1], reverse=True)

ownerless_critical = [
    (doc_id, degree) for doc_id, degree in ownerless_with_degree
    if degree >= np.percentile(degree_values, 75)  # Top 25% by degree
]

if len(ownerless_critical) > 0:
    print(f"Found {len(ownerless_critical)} ownerless critical documents (top 25% by degree):")
    print(f"\n{'Document ID':<12} {'Degree':<8} {'Name'}")
    print("-" * 60)
    for doc_id, degree in ownerless_critical:
        doc_data = documents_df[documents_df['document_id'] == doc_id].iloc[0]
        name = doc_data['name'][:45]
        print(f"{doc_id:<12} {degree:<8} {name}")
    print(f"\n⚠️ ACCOUNTABILITY RISK: Assign owners immediately")
else:
    print("No ownerless critical documents found.")

In [ ]:
# Anomaly 3: Isolated clusters (orphaned documentation)
print("\nANOMALY 3: Isolated Clusters")
print("Groups of documents that only reference each other, not connected to main network.")
print("This indicates siloed documentation: potential integration opportunities.\n")

# Find non-singleton components
small_components = [c for c in components if 2 <= len(c) <= 20]

if len(small_components) > 0:
    print(f"Found {len(small_components)} small isolated cluster(s):")
    for i, component in enumerate(small_components):
        print(f"\n  Cluster {i+1}: {len(component)} documents")
        component_docs = [d for d in component if d in documents_df['document_id'].values]
        for doc_id in list(component_docs)[:5]:  # Show first 5
            doc_data = documents_df[documents_df['document_id'] == doc_id].iloc[0]
            print(f"    - {doc_id}: {doc_data['name'][:50]}")
        if len(component_docs) > 5:
            print(f"    ... and {len(component_docs) - 5} more")
    print(f"\n⚠️ INTEGRATION OPPORTUNITY: Review and potentially merge clusters")
else:
    print("No isolated clusters found.")

In [ ]:
# Anomaly 4: Dense duplicate clusters (over-referencing)
print("\nANOMOLY 4: Dense Duplicate Clusters")
print("Groups of documents that heavily reference each other (possible duplicates).")
print("This indicates data quality issue: redundant or over-referenced documentation.\n")

# Find subgraphs with high edge density
from itertools import combinations

def find_dense_subgraph(G, min_size=5, min_density=0.5):
    """Find subgraphs with high internal density."""
    dense_subgraphs = []
    for component in nx.connected_components(G):
        subG = G.subgraph(component)
        if len(component) >= min_size and nx.density(subG) >= min_density:
            dense_subgraphs.append((component, nx.density(subG)))
    return dense_subgraphs

dense_subgraphs = find_dense_subgraph(G, min_size=5, min_density=0.4)

if len(dense_subgraphs) > 0:
    print(f"Found {len(dense_subgraphs)} dense subgraph(s):")
    for i, (component, density) in enumerate(dense_subgraphs):
        component_docs = [d for d in component if d in documents_df['document_id'].values]
        print(f"\n  Cluster {i+1}: {len(component_docs)} documents, density={density:.3f}")
        for doc_id in list(component_docs)[:5]:
            doc_data = documents_df[documents_df['document_id'] == doc_id].iloc[0]
            print(f"    - {doc_id}: {doc_data['name'][:50]}")
        if len(component_docs) > 5:
            print(f"    ... and {len(component_docs) - 5} more")
    print(f"\n⚠️ QUALITY ISSUE: Review for duplicates or merge candidates")
else:
    print("No dense clusters found (this is expected - most networks are sparse).")

## 5. Subgraph Analysis

Extract and analyze specific parts of the network for focused investigation.

In [ ]:
# Extract subgraph: high-degree nodes and their neighbors
print("Subgraph 1: High-Degree Nodes and Neighbors")
print("Zooming into the most connected part of the network.\n")

high_degree_threshold = np.percentile(degree_values, 90)  # Top 10%
high_degree_doc_nodes = [
    node for node in G.nodes()
    if node in documents_df['document_id'].values and degrees.get(node, 0) >= high_degree_threshold
]

# Get neighbors
neighbors = set()
for node in high_degree_doc_nodes:
    neighbors.update(G.neighbors(node))

high_degree_subgraph_nodes = set(high_degree_doc_nodes) | neighbors
G_high_degree = G.subgraph(high_degree_subgraph_nodes).copy()

print(f"High-degree threshold (90th percentile): {high_degree_threshold:.1f}")
print(f"High-degree nodes: {len(high_degree_doc_nodes)}")
print(f"Subgraph size: {G_high_degree.number_of_nodes()} nodes, {G_high_degree.number_of_edges()} edges")
print(f"Subgraph density: {nx.density(G_high_degree):.4f}")

# Analyze status distribution in subgraph
status_dist = {}
for node in G_high_degree.nodes():
    if node in documents_df['document_id'].values:
        status = documents_df[documents_df['document_id'] == node]['status'].iloc[0]
        status_dist[status] = status_dist.get(status, 0) + 1

print(f"\nDocument status in subgraph:")
for status, count in sorted(status_dist.items()):
    print(f"  {status}: {count}")

In [ ]:
# Extract subgraph: single team's dependencies
print("\nSubgraph 2: Team Dependencies")
print("Analyzing what documents a specific team depends on.\n")

# Pick first team
sample_team = teams_df.iloc[0]['team_id']
sample_team_name = teams_df.iloc[0]['name']

# Find all documents used by this team
team_docs = usages_df[usages_df['team_id'] == sample_team]['document_id'].values
team_neighbors = set()
for doc in team_docs:
    if doc in G:
        team_neighbors.update(G.neighbors(doc))

team_subgraph_nodes = set(team_docs) | team_neighbors | {sample_team}
G_team = G.subgraph(team_subgraph_nodes).copy()

print(f"Team: {sample_team_name} ({sample_team})")
print(f"Documents directly used: {len(team_docs)}")
print(f"Subgraph size: {G_team.number_of_nodes()} nodes, {G_team.number_of_edges()} edges")

# Find critical documents (high degree within team subgraph)
team_subgraph_degrees = dict(G_team.degree())
critical_docs = sorted(
    [(doc, team_subgraph_degrees[doc]) for doc in team_docs if doc in team_subgraph_degrees],
    key=lambda x: x[1],
    reverse=True
)[:5]

print(f"\nTop 5 critical documents for {sample_team_name}:")
print(f"{'Document ID':<12} {'Degree':<8} {'Name'}")
print("-" * 60)
for doc_id, degree in critical_docs:
    doc_data = documents_df[documents_df['document_id'] == doc_id].iloc[0]
    print(f"{doc_id:<12} {degree:<8} {doc_data['name'][:45]}")

print(f"\n⚠️ These documents are single points of failure for {sample_team_name}")

## 6. Exercises

Practice profiling to discover insights about your own graphs.

### Exercise 1: Build a Profiling Checklist

Create a function that generates a complete profiling report including all key metrics and anomaly flags.

In [7]:
# SOLUTION: Exercise 1

def profile_and_flag(G, documents_df=None, percentile_high_degree=90, percentile_low_degree=10):
    """
    Complete profiling with anomaly flags.
    
    Returns: dict with metrics and flags
    """
    profile = {
        'metrics': graph_profile(G, verbose=False),
        'anomalies': {}
    }
    
    # Check for isolates
    profile['anomalies']['has_isolates'] = profile['metrics']['isolates'] > 0
    profile['anomalies']['num_isolates'] = profile['metrics']['isolates']
    
    # Check for multiple components
    profile['anomalies']['fragmented'] = profile['metrics']['components'] > 1
    profile['anomalies']['num_components'] = profile['metrics']['components']
    
    # Check for low density
    profile['anomalies']['sparse_network'] = profile['metrics']['density'] < 0.05
    
    # Check for high degree variance
    degrees = dict(G.degree())
    degree_values = list(degrees.values())
    profile['anomalies']['high_degree_variance'] = np.std(degree_values) > 2 * np.mean(degree_values)
    
    return profile

# Run profiling
profile_with_flags = profile_and_flag(G, documents_df)

print("Comprehensive Profiling Report:")
print(f"\nMetrics:")
for metric, value in profile_with_flags['metrics'].items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value}")

print(f"\nAnomalies & Flags:")
for flag, value in profile_with_flags['anomalies'].items():
    symbol = '⚠️ ' if value else '✓ '
    print(f"  {symbol}{flag}: {value}")

Comprehensive Profiling Report:

Metrics:
  nodes: 108
  edges: 490
  density: 0.0848
  avg_degree: 9.0741
  min_degree: 1
  max_degree: 35
  components: 1
  isolates: 0
  avg_clustering: 0.1951

Anomalies & Flags:
  ✓ has_isolates: False
  ✓ num_isolates: 0
  ✓ fragmented: False
  ⚠️ num_components: 1
  ✓ sparse_network: False
  ✓ high_degree_variance: False


### Exercise 2: Find At-Risk Documents

Identify documents that are critical but have vulnerability indicators (no owner, deprecated status, high degree).

In [ ]:
# SOLUTION: Exercise 2

def find_at_risk_documents(G, documents_df, degree_threshold_percentile=75):
    """
    Find documents that are critical but vulnerable.
    Criteria: high degree AND (no owner OR deprecated)
    """
    degrees = dict(G.degree())
    degree_threshold = np.percentile(
        [d for d in degrees.values() if d > 0],
        degree_threshold_percentile
    )
    
    at_risk = []
    for idx, row in documents_df.iterrows():
        doc_id = row['document_id']
        degree = degrees.get(doc_id, 0)
        is_critical = degree >= degree_threshold
        is_risky = pd.isna(row['owner_id']) or row['status'] == 'deprecated'
        
        if is_critical and is_risky:
            at_risk.append({
                'document_id': doc_id,
                'name': row['name'],
                'degree': degree,
                'status': row['status'],
                'owner_id': row['owner_id'],
                'risk_factors': []
            })
            
            # Annotate risk factors
            if pd.isna(row['owner_id']):
                at_risk[-1]['risk_factors'].append('No owner')
            if row['status'] == 'deprecated':
                at_risk[-1]['risk_factors'].append('Deprecated')
    
    return pd.DataFrame(at_risk)

at_risk_df = find_at_risk_documents(G, documents_df, degree_threshold_percentile=75)
print(f"At-Risk Documents: {len(at_risk_df)}\n")
print(at_risk_df[['document_id', 'degree', 'status', 'owner_id']].to_string(index=False))
print(f"\nRecommendation: Prioritize remediation of these {len(at_risk_df)} documents")

### Exercise 3: Component Analysis

Analyze the largest connected component vs isolated nodes. What fraction of the graph is connected?

In [ ]:
# SOLUTION: Exercise 3

def analyze_connectivity(G, documents_df):
    """
    Analyze graph connectivity: largest component, isolates, fragmentation.
    """
    components = list(nx.connected_components(G))
    doc_components = [
        [d for d in c if d in documents_df['document_id'].values]
        for c in components
    ]
    
    # Filter out empty components (all teams)
    doc_components = [c for c in doc_components if len(c) > 0]
    doc_components.sort(key=len, reverse=True)
    
    largest_component = doc_components[0] if doc_components else []
    isolates = [c[0] for c in doc_components if len(c) == 1]
    
    total_docs = len(documents_df)
    connected_docs = len(largest_component)
    isolated_docs = len(isolates)
    
    connectivity_metrics = {
        'total_documents': total_docs,
        'num_components': len(doc_components),
        'largest_component_size': connected_docs,
        'largest_component_pct': 100 * connected_docs / total_docs,
        'isolated_documents': isolated_docs,
        'isolated_pct': 100 * isolated_docs / total_docs,
        'fragmentation_ratio': len(doc_components) / total_docs,  # Components per document
    }
    
    return connectivity_metrics

metrics = analyze_connectivity(G, documents_df)
print("Connectivity Analysis:")
print(f"  Total documents: {metrics['total_documents']}")
print(f"  Number of components: {metrics['num_components']}")
print(f"  Largest component: {metrics['largest_component_size']} docs ({metrics['largest_component_pct']:.1f}%)")
print(f"  Isolated documents: {metrics['isolated_documents']} ({metrics['isolated_pct']:.1f}%)")
print(f"  Fragmentation ratio: {metrics['fragmentation_ratio']:.4f}")

if metrics['largest_component_pct'] > 80:
    print(f"\n✓ GOOD: Most documents are in the main connected component")
elif metrics['largest_component_pct'] > 50:
    print(f"\n⚠️ FAIR: Document network is somewhat fragmented")
else:
    print(f"\n⚠️ POOR: Document network is highly fragmented")

## 7. Key Takeaways

### Why Profile Graphs?

1. **Catch data quality issues early** — Profiling reveals obsolete docs, orphaned clusters, duplicates
2. **Understand network structure** — Metrics (density, components, clustering) tell you the story
3. **Avoid algorithm pitfalls** — Algorithms assume certain properties; profiling validates assumptions
4. **Find vulnerabilities** — Isolated nodes, high-degree hubs, fragmentation reveal risks

### Essential Metrics

| Metric | What it tells you | Red flags |
|--------|-----------------|------------|
| **Density** | How connected is the graph? | Very low (<0.01) = sparse, fragmented |
| **Degree distribution** | How are connections spread? | High variance = power-law, has hubs |
| **Components** | How fragmented is the graph? | Many components = siloed data |
| **Isolates** | Are there disconnected nodes? | Yes = data quality or new items |
| **Clustering** | Do triangles form communities? | High = tightly coupled nodes |
| **Assortativity** | Do similar nodes connect? | High = homophily, birds of a feather |

### Anomalies This Lesson Found

1. **Obsolete but used documents** → Governance debt, migration needed
2. **Ownerless critical documents** → Accountability risk, assign owners
3. **Isolated clusters** → Siloed documentation, integration opportunities
4. **Dense duplicates** → Data quality issue, review for redundancy

### Next Steps

**Lesson 04 — Centrality Algorithms**: Once your graph is profiled and cleaned, compute centrality to find the most important nodes. Metrics alone won't tell you *why* nodes matter; algorithms quantify importance under different definitions.

---

**Congratulations!** You now know how to systematically inspect graphs before trusting algorithm results. You've discovered that profiling catches data quality issues and governance risks that algorithms alone would miss.

**Challenge for you**: Apply this profiling workflow to one of your own graphs. You'll likely discover issues you didn't know existed.